# 2) Logistic Regression with more hidden layers

In [12]:
import torch
import torch.nn as nn

from collections.abc import Sequence           # to connect hidden layers sequentially

## NN with more hidden layers, explicit version

In [13]:
class TorchMLP_explicit(nn.Module):

  #3 LINEAR LAYERS + 1 ACTIVATION FUNCTION ReLu
  def __init__(self, input_dim, hidden_dim_1, hidden_dim_2, hidden_dim_3, output_dim):

    super().__init__()

    self.linear1 = nn.Linear(input_dim, hidden_dim_1)
    self.linear2 = nn.Linear(hidden_dim_1, hidden_dim_2)
    self.linear3 = nn.Linear(hidden_dim_2, hidden_dim_3)
    self.linear4 = nn.Linear(hidden_dim_3, output_dim)
    self.relu = nn.ReLU()

  #DEFINE FORWARD PHASE
  def forward(self, x):
    x = self.linear1(x)
    x = self.linear2(x)
    x = self.linear3(x)
    x = self.relu(x)
    return self.linear4(x)



In [14]:

model_exp = TorchMLP_explicit(784, 100, 200, 100, 10)

## NN with more hidden layers, implicit version

Fast method to define hidden layers

In [15]:

class TorchMLP(nn.Module):
    
    def __init__(self, input_dim: int, hidden_dims: Sequence[int], output_dim: int):
        
        super().__init__()

        dims = [input_dim, *hidden_dims, output_dim]                   # list of layers dimensions / number of units

        layers = []                                                    # contains fully connections of layers / arrows

        for i, (in_dim, out_dim) in enumerate(zip(dims, dims[1:])):    # enumerate: run on couples
                                                                       # zip : pairs each couple
                                                                       # [1: ] : takes off element 0
                                                                       # connect adjacent layers
            
            layers.append(nn.Linear(in_dim, out_dim))                  # linearly

            layers.append(nn.ReLU())                                   # add ReLu after each hidden layer

            layers = layers[:-1]                                       # remove ReLu applied to output


        self.model = nn.Sequential(*layers)                            # run layers sequentially

    def forward(self, x):                                              # forward computations sequential
        
        return self.model(x)


In [22]:
#CALL THE MODEL

hidden_layers_dim = [100, 200, 100]

model_lr = TorchMLP(784, hidden_layers_dim, 10)

# LOSS

loss_lr = nn.CrossEntropyLoss()


## Comparison of optimization methods

We consider the NN model_lr, with cross entropy loss.

**OPTIMIZATION PHASE:** an optimization algorithm minimizes the loss function $L$ over the weights and updates the weights as:
$$
W_{i} \leftarrow W_{i} - \nu \cdot \nabla_{W_{i}}(L),
$$
where $\nu$ is the learning rate.

The gradient is subtracted from the weights with a step of $\nu$ at each iteration of the algorithm.

### Gradient Descent

> Classic version, where $\nu > 0$ is constant (and small).

### Stochastic Gradient Descent (SDG)

### Adam

> Adaptative method, where the learning rate is not constant and updated at every iteration depending on the gradient size. If the gradient is large, the learning rate decreases. If the gradient is small, the learning rate increases.

> Adam avoids that during the minimization process, the loss gets stucked in false minima or flat regions.

In [21]:
#OPTIMIZER Adam

optimizer = optim.Adam(model.parameters())


NameError: name 'optim' is not defined

**Adam** is a gradient optimization method, where at each iteration the learning rate is recomputed depending on the size of the gradient. If the gradient is big, the learning rate decreases. If the gradient is small, the learning rate increases. This allows the gradient to not being stuck into fake global minima, valleys and flat regions.

**Epoch** = one round on all training data / one round on which the algorithm sees all the training data

In [ ]:
torch_mlp.cuda()
torch_X_train = torch_X_train.to('cuda')
torch_y_train = torch_y_train.to('cuda')
torch_X_valid = torch_X_valid.to('cuda')
torch_y_valid = torch_y_valid.to('cuda')


for epoch in range(1000):                         #RUN FOR 1000 EPOCHS: see all data 1000 times
  optimizer.zero_grad()
  y_pred = torch_mlp(torch_X_train)               #FORWARD PHASE: output y_pred computed
                                                  #-> the activation function is already included
  loss_result = loss(y_pred, torch_y_train)       #LOSS COMPUTATION: L(y_pred, y_train)
  loss_result.backward()                          #BACKWARD PHASE: gradient of loss over the weights is computed: dL(w)/dw
  optimizer.step()                                #OPTIMIZATION PHASE: update weights: w <- w + (learning_rate)*(gradient)


  if epoch % 10 == 0:                                     #EVALUATION PHASE: every 10 epochs
    with torch.no_grad():
      pred_valid = torch_mlp(torch_X_valid.to('cuda'))

    _, prediction = torch.ax(pred_valid, dim=1)
    print(accuracy_score(y_valid, prediction.to('cpu')))


_IncompleteInputError: incomplete input (2647271325.py, line 19)

In [ ]:
for param in model.parameters():
  print(param.numel())

print('Total : ', sum(param.numel() for param in model.parameters() ))
